# Agente com multiplas ferramentas e memoria

Este notebook estende o harness do zero (projeto 1) em duas direcoes:

1. **Multiplas ferramentas de tipos diferentes**: calculadora, RAG sobre documentos internos (reaproveitando a Fase 4) e busca na web
2. **Memoria entre perguntas na mesma sessao**: memoria de **curto prazo** (o historico da conversa cresce a cada turno, em vez de comecar do zero) e memoria de **longo prazo** (fatos explicitos que o usuario pede para o agente lembrar, injetados no system prompt em turnos futuros)

**Pre-requisito**: harness do zero (projeto 1) -- este notebook reaproveita o `run_agent` e o dispatcher de la, so adicionando ferramentas e uma camada de sessao por cima.

**Sobre os testes**: como nos notebooks anteriores, a mecanica (ferramentas, dispatcher, loop, memoria) e validada aqui com um **cliente mock**, sem precisar de chave de API. A secao final mostra a mesma coisa plugada na API real.

## 1. Ferramenta 1: calculadora (reaproveitada do projeto 1)

In [ ]:
import ast
import operator as op

_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}

def _eval_node(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Expressao nao suportada: {ast.dump(node)}")

def calculator(expression):
    try:
        return {"result": _eval_node(ast.parse(expression, mode="eval").body)}
    except Exception as e:
        return {"error": f"Nao foi possivel avaliar \'{expression}\': {e}"}


## 2. Ferramenta 2: RAG sobre documentos internos (reaproveitando a Fase 4)

Mesma logica de chunking + busca por TF-IDF construida no notebook de RAG. Aqui ela vira uma **ferramenta que o agente decide quando chamar**, em vez de um pipeline fixo -- essa e exatamente a diferenca mencionada no final do notebook de RAG: o agente pode reformular a pergunta e tentar de novo se a primeira busca nao trouxe informacao suficiente.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

documents = [
    {"id": "politica_ferias", "text": "Politica de ferias da empresa Aurora Tech. Todo colaborador tem direito a 30 dias corridos de ferias por ano trabalhado. As ferias devem ser solicitadas com no minimo 30 dias de antecedencia ao gestor direto."},
    {"id": "politica_reembolso", "text": "Politica de reembolso de despesas da Aurora Tech. Despesas com viagens a trabalho devem ser reembolsadas em ate 15 dias uteis apos o envio da nota fiscal. O limite diario para alimentacao em viagem e de R$ 120,00."},
    {"id": "politica_home_office", "text": "Politica de trabalho remoto da Aurora Tech. O auxilio home office e de R$ 150,00 mensais, pago junto com o salario."},
]

def chunk_text(text, chunk_size=25, overlap=5):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = end - overlap
    return chunks

all_chunks = []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({"doc_id": doc["id"], "chunk_id": f"{doc['id']}_{i}", "text": chunk})

chunk_texts = [c["text"] for c in all_chunks]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

def buscar_politica_interna(pergunta, top_k=2):
    q_vec = vectorizer.transform([pergunta])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    resultados = [
        {"chunk_id": all_chunks[i]["chunk_id"], "texto": all_chunks[i]["text"], "score": round(float(sims[i]), 3)}
        for i in top_idx
    ]
    return {"resultados": resultados}


print(buscar_politica_interna("quantos dias de ferias tenho direito?"))


## 3. Ferramenta 3: busca na web

Aqui usamos um indice local fixo, **so para o notebook rodar offline** -- na versao real, isso seria uma chamada a uma API de busca (Tavily, SerpAPI, Brave Search, ou a ferramenta de busca web nativa da Anthropic). A interface (recebe uma query, devolve um resultado) e a mesma; so o corpo da funcao mudaria.

In [ ]:
_FAKE_WEB_INDEX = {
    "python 3.13": "Python 3.13 foi lancado em outubro de 2024, com melhorias de performance no interpretador.",
    "anthropic claude": "A Anthropic e uma empresa de pesquisa em IA que desenvolve a familia de modelos Claude.",
}

def buscar_na_web(query):
    """Mock local. Para usar busca real, troque por uma chamada a API de busca de sua preferencia."""
    key = query.strip().lower()
    for k, v in _FAKE_WEB_INDEX.items():
        if k in key or key in k:
            return {"resultado": v}
    return {"resultado": "Nenhum resultado encontrado (indice de demonstracao limitado)."}


## 4. Dispatcher e loop (reaproveitados do projeto 1, com 3 ferramentas em vez de 2)

In [ ]:
import json

TOOL_FUNCTIONS = {
    "calculator": lambda args: calculator(args["expression"]),
    "buscar_politica_interna": lambda args: buscar_politica_interna(args["pergunta"]),
    "buscar_na_web": lambda args: buscar_na_web(args["query"]),
}

def execute_tool(tool_name, tool_args):
    if tool_name not in TOOL_FUNCTIONS:
        return {"error": f"ferramenta desconhecida: {tool_name}"}
    try:
        return TOOL_FUNCTIONS[tool_name](tool_args)
    except Exception as e:
        return {"error": f"erro ao executar {tool_name}: {e}"}


def run_agent_step(client, messages, max_steps=5, verbose=True):
    """Identico ao run_agent do projeto 1 -- roda ate o modelo parar de pedir ferramentas ou atingir max_steps."""
    steps = 0
    while steps < max_steps:
        steps += 1
        response = client.create(messages=messages)
        text_blocks = [b for b in response.content if b.type == "text"]
        tool_blocks = [b for b in response.content if b.type == "tool_use"]

        if verbose:
            for b in text_blocks:
                print(f"  [passo {steps}] modelo: {b.text}")

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return " ".join(b.text for b in text_blocks)

        tool_results = []
        for block in tool_blocks:
            if verbose:
                print(f"  [passo {steps}] ferramenta: {block.name}({block.input})")
            result = execute_tool(block.name, block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result, ensure_ascii=False),
            })
        messages.append({"role": "user", "content": tool_results})

    return None


## 5. A classe `Session`: memoria de curto e longo prazo

Duas coisas novas em relacao ao projeto 1:

- **Memoria de curto prazo**: `self.messages` **nao e reiniciada a cada chamada** -- ela cresce ao longo da sessao inteira, entao o agente "lembra" o que foi dito em turnos anteriores dentro da mesma conversa (isso e, na pratica, so o context window do modelo sendo reaproveitado entre turnos)
- **Memoria de longo prazo**: um dicionario `long_term_memory` guarda fatos extraidos explicitamente (aqui, com uma heuristica simples de regex -- em um sistema real, isso normalmente seria feito com uma chamada de LLM dedicada para extrair fatos, ou com um RAG sobre um banco de "memorias" como mencionado na Fase 5). Esses fatos sao injetados no **system prompt** a cada novo turno, entao persistem mesmo que a conversa fique tao longa que teoricamente sairia do contexto

In [ ]:
import re

class Session:
    def __init__(self, client):
        self.client = client
        self.messages = []          # memoria de curto prazo: historico da sessao atual
        self.long_term_memory = {}  # memoria de longo prazo: fatos explicitos

    def _system_prompt(self):
        base = "Voce e um assistente com acesso a calculadora, busca em politicas internas da empresa e busca na web."
        if self.long_term_memory:
            fatos = "; ".join(f"{k}: {v}" for k, v in self.long_term_memory.items())
            base += f" Fatos conhecidos sobre o usuario: {fatos}."
        return base

    def _extrair_fato(self, user_input):
        """Heuristica simples de extracao. Em producao, considere usar uma chamada de LLM dedicada para isso."""
        m = re.search(r"lembre que (.+?) e (.+)", user_input, re.IGNORECASE)
        if m:
            self.long_term_memory[m.group(1).strip()] = m.group(2).strip().rstrip(".")
        m2 = re.search(r"meu nome e (.+)", user_input, re.IGNORECASE)
        if m2:
            self.long_term_memory["nome do usuario"] = m2.group(1).strip().rstrip(".")

    def chat(self, user_input, verbose=True):
        self._extrair_fato(user_input)
        self.messages.append({"role": "user", "content": user_input})
        resposta = run_agent_step(self.client, self.messages, verbose=verbose)
        return resposta


## 6. Testando com um cliente mock (conversa de 3 turnos)

Cenario: o usuario se apresenta (turno 1), faz uma pergunta que usa a ferramenta de RAG (turno 2), e depois pergunta algo que so pode ser respondido se o agente **lembrar** o que foi dito no turno 1 (turno 3) -- isso testa memoria de curto prazo (o historico da conversa) e de longo prazo (o fato extraido e reinjetado).

In [ ]:
class FakeToolUseBlock:
    def __init__(self, id, name, input_):
        self.type = "tool_use"; self.id = id; self.name = name; self.input = input_

class FakeTextBlock:
    def __init__(self, text):
        self.type = "text"; self.text = text

class FakeResponse:
    def __init__(self, content, stop_reason):
        self.content = content; self.stop_reason = stop_reason

class FakeAnthropicClient:
    def __init__(self, scripted_responses):
        self.scripted_responses = scripted_responses
        self.call_count = 0
    def create(self, **kwargs):
        response = self.scripted_responses[self.call_count]
        self.call_count += 1
        return response


scripted = [
    FakeResponse([FakeTextBlock("Prazer, Joao! Como posso ajudar?")], "end_turn"),
    FakeResponse([FakeToolUseBlock("t1", "buscar_politica_interna", {"pergunta": "quantos dias de ferias"})], "tool_use"),
    FakeResponse([FakeTextBlock("Voce tem direito a 30 dias corridos de ferias por ano.")], "end_turn"),
    FakeResponse([FakeTextBlock("Seu nome e Joao, como voce me disse anteriormente.")], "end_turn"),
]
client = FakeAnthropicClient(scripted)
session = Session(client)

print("Turno 1:")
r1 = session.chat("Ola, meu nome e Joao.")
print("Resposta:", r1)
print("Memoria de longo prazo:", session.long_term_memory)

print("\nTurno 2:")
r2 = session.chat("Quantos dias de ferias eu tenho direito?")
print("Resposta:", r2)

print("\nTurno 3 (depende de lembrar o nome do turno 1):")
r3 = session.chat("Qual e o meu nome?")
print("Resposta:", r3)

assert session.long_term_memory.get("nome do usuario") == "Joao"
print("\nMemoria persistida corretamente entre turnos.")


## 7. A mesma sessao com a API real

A `Session` nao muda: so trocamos o cliente mock pelo cliente real, e agora `_system_prompt()` e efetivamente enviado como o parametro `system` da API (nos testes acima, o mock ignorava isso, ja que as respostas eram roteirizadas).

**Requer**: `pip install anthropic` e `ANTHROPIC_API_KEY`.

In [ ]:
import os
from anthropic import Anthropic

TOOLS_SCHEMA = [
    {
        "name": "calculator",
        "description": "Avalia uma expressao aritmetica simples.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
    {
        "name": "buscar_politica_interna",
        "description": "Busca informacoes nas politicas internas da empresa Aurora Tech (ferias, reembolso, home office).",
        "input_schema": {
            "type": "object",
            "properties": {"pergunta": {"type": "string"}},
            "required": ["pergunta"],
        },
    },
    {
        "name": "buscar_na_web",
        "description": "Busca informacoes gerais na internet.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
]

raw_client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

class RealSession(Session):
    """Mesma logica de Session, mas monta a chamada real -- incluindo o system prompt dinamico com memoria de longo prazo."""
    def chat(self, user_input, verbose=True):
        self._extrair_fato(user_input)
        self.messages.append({"role": "user", "content": user_input})

        class _ClientWrapper:
            def create(inner_self, messages):
                return raw_client.messages.create(
                    model="claude-sonnet-4-6",
                    max_tokens=1024,
                    system=self._system_prompt(),   # aqui a memoria de longo prazo entra de fato
                    tools=TOOLS_SCHEMA,
                    messages=messages,
                )

        return run_agent_step(_ClientWrapper(), self.messages, verbose=verbose)


real_session = RealSession(client=None)  # client nao usado diretamente, ver _ClientWrapper acima
print(real_session.chat("Ola, meu nome e Joao. Lembre que meu time e Engenharia."))
print(real_session.chat("Quantos dias de ferias eu tenho direito?"))
print(real_session.chat("Qual e o meu nome e em qual time eu trabalho?"))


## Notas finais

- **Memoria de longo prazo real**: a heuristica de regex aqui e deliberadamente simples, so para ilustrar o mecanismo. Em um sistema de producao, a extracao de fatos costuma ser feita com uma chamada de LLM dedicada ("extraia fatos relevantes desta conversa para lembrar depois"), e o armazenamento costuma ser um RAG sobre um banco de memorias (Fase 4) em vez de um dicionario em memoria que se perde quando o processo termina
- **Memoria persistente entre sessoes** (nao so dentro da mesma sessao, como aqui): exigiria salvar `long_term_memory` em disco ou banco de dados entre execucoes do notebook -- o mecanismo de injetar no system prompt continua o mesmo
- **RAG como ferramenta vs. pipeline fixo**: repare que o agente decide *se* e *quando* chamar `buscar_politica_interna` -- ele poderia, em teoria, chamar a ferramenta de novo com uma pergunta reformulada se a primeira busca nao trouxe informacao suficiente (algo que um pipeline RAG fixo, como o da Fase 4, nao faz sozinho)
- **Proximo projeto da fase**: harness de avaliacao de agente -- agora que o agente tem multiplos passos e memoria, medir taxa de sucesso, numero de passos e custo por trajetoria completa (nao so por resposta unica) fica mais interessante e mais representativo de um agente real
